# 03 · DentVoxel 3D CBCT WSP Benchmark (Validation Only)

Evaluates **Track 1 (`dentvoxel_cls`: scan-split axial maxillary sinus presence, `cls-lenet2d-v1`)** and **Track 2 (`dentvoxel_seg`: 3D tooth union, `cbct-wsp82-v1`, 82 params)** on the 100-scan NIfTI DentVoxel CBCT dataset.

**First step:** Run Cell 2 to execute the mandatory **HU intensity calibration audit** before locking final CBCT prompts. Held-out test scans remain sealed.

In [ ]:
from pathlib import Path
import json
import sys

REPO_DIR = Path("/content/dental-wsp")
if REPO_DIR.is_dir() and str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

DATA_DIR = Path("/content/drive/MyDrive/Colab Notebooks/WSP_Work/dentvoxel")
from benchmarks.dentvoxel.adapter import verify_dentvoxel_directory
from wsp_core.guards import exposure_guard

exposure_guard(DATA_DIR)
audit = verify_dentvoxel_directory(DATA_DIR, max_scans=10)
print(json.dumps(audit, indent=2))

In [ ]:
import pandas as pd
from benchmarks.dentvoxel.adapter import load_dentvoxel_cls, load_dentvoxel_seg
from wsp_core.evaluation.classification import evaluate_classifier
from wsp_core.evaluation.segmentation import evaluate_segmentation_paired_audit, evaluate_segmentation_swaps
from wsp_core.schemas import strict_json_loads

SPLIT_FILE = DATA_DIR / "splits.private.json"
NORM_MODE = audit["intensity_audit"]["recommended_mode"]
SEG_LOCK = Path("/content/dental-wsp/benchmarks/dentvoxel/locks/dentvoxel_seg.responses.lock.json")
CLS_LOCK = Path("/content/dental-wsp/benchmarks/dentvoxel/locks/dentvoxel_cls.responses.lock.json")

if SEG_LOCK.is_file():
    val_seg = load_dentvoxel_seg(DATA_DIR, SPLIT_FILE, split="val", norm_mode=NORM_MODE)
    lock = strict_json_loads(SEG_LOCK.read_bytes())
    rows = []
    for slot, rec in lock["attempts"].items():
        if rec.get("status") != "valid":
            continue
        sw = evaluate_segmentation_swaps(rec["parameters"], val_seg)
        au = evaluate_segmentation_paired_audit(rec["parameters"], val_seg)
        rows.append({
            "Reply": slot,
            "Intact Dice (%)": sw["intact"]["correct_dice_pct"],
            "Swap loss (pp)": sw["intact"]["swap_loss_pp"],
            "Outside FP share (%)": au["outside_fp_reduction_share_pct"],
            "3D Box IoU A (%)": au["box_iou_a_pct"],
            "Excluded tooth voxels A (%)": au["excluded_tooth_a_pct"],
        })
    display(pd.DataFrame(rows).set_index("Reply").round(4))

if CLS_LOCK.is_file():
    val_cls = load_dentvoxel_cls(DATA_DIR, SPLIT_FILE, split="val", norm_mode=NORM_MODE)
    lock = strict_json_loads(CLS_LOCK.read_bytes())
    rows = []
    for slot, rec in lock["attempts"].items():
        if rec.get("status") != "valid":
            continue
        res = evaluate_classifier(rec["parameters"], val_cls)
        rows.append({"Reply": slot, **res})
    display(pd.DataFrame(rows).set_index("Reply").round(4))